In [1]:
import re

CHAPTER_TITLE_DE = "2.1 Definition und Konzepte des Dynamic Pricing im stationären Einzelhandel"
CHAPTER_TEXT_DE = """
Die theoretischen Grundlagen des Dynamic Pricing werden detailliert beschrieben.
Es wird erläutert, wie Dynamic Pricing im Einzelhandel funktioniert und welche unterschiedlichen Ansätze es gibt,
z. B. zeitabhängige Preise oder an die Nachfrage gekoppelte Preisänderungen.
Es wird nicht auf operative Details der Preisgestaltung oder spezifische Implementierungsstrategien eingegangen.
Der Online-Handel wird explizit ausgeklammert.
""".strip()

# Simple, explicit English scope translation (we'll automate DE/EN later)
CHAPTER_SCOPE_EN = (
    "Conceptual and theoretical foundations of dynamic pricing in brick-and-mortar (in-store/offline) retail. "
    "Explain mechanisms and approaches (e.g., time-dependent pricing, demand-linked price changes). "
    "Do not focus on operational implementation details; focus on definitions, concepts, and approaches."
)

def normalize_query(q: str) -> str:
    q = q.lower()
    q = re.sub(r"[^a-z0-9\s\"-]", " ", q)
    q = re.sub(r"\s+", " ", q).strip()
    return q

# Query variants: 1 broad + multiple focused (no hard exclusions yet)
QUERIES = {
    "broad": normalize_query(
        'dynamic pricing definition concepts brick-and-mortar retail in-store offline retail'
    ),
    "theory_foundations": normalize_query(
        'dynamic pricing theory conceptual framework retail'
    ),
    "time_based": normalize_query(
        'time-based pricing time-dependent pricing dynamic pricing retail store'
    ),
    "demand_based": normalize_query(
        'demand-based pricing demand-driven pricing dynamic pricing retail store'
    ),
    "revenue_management": normalize_query(
        'revenue management pricing retail dynamic pricing in-store'
    ),
    "price_discrimination": normalize_query(
        'price discrimination dynamic pricing retail store'
    ),
}

QUERY_LIST = list(QUERIES.values())

print("Chapter (EN scope):", CHAPTER_SCOPE_EN)
print("\nQueries:")
for k, v in QUERIES.items():
    print(f"- {k}: {v}")


Chapter (EN scope): Conceptual and theoretical foundations of dynamic pricing in brick-and-mortar (in-store/offline) retail. Explain mechanisms and approaches (e.g., time-dependent pricing, demand-linked price changes). Do not focus on operational implementation details; focus on definitions, concepts, and approaches.

Queries:
- broad: dynamic pricing definition concepts brick-and-mortar retail in-store offline retail
- theory_foundations: dynamic pricing theory conceptual framework retail
- time_based: time-based pricing time-dependent pricing dynamic pricing retail store
- demand_based: demand-based pricing demand-driven pricing dynamic pricing retail store
- revenue_management: revenue management pricing retail dynamic pricing in-store
- price_discrimination: price discrimination dynamic pricing retail store


# OpenAlex (Works) — broad scholarly coverage (articles/books/chapters)

In [4]:
import os
import time
import requests
import pandas as pd
from typing import Any, Dict, List, Optional

OPENALEX_API_KEY = os.getenv("OPENALEX_API_KEY", "WOZ9qphHIT86mmH5BQFKuB")  # get one at openalex.org/settings/api
if not OPENALEX_API_KEY:
    raise RuntimeError("Set OPENALEX_API_KEY env var.")

BASE_URL = "https://api.openalex.org/works"

PER_PAGE = 100              # up to 200; with abstracts, 50-100 is usually safer
MAX_WORKS_PER_QUERY = 400   # set None to fetch "all" (can be a lot)
TIMEOUT_SEC = 30

OUTPUT_CSV_PATH = "openalex_works.csv"

# ----------------------------
# Helpers
# ----------------------------
def abstract_from_inverted_index(inv: Optional[Dict[str, List[int]]]) -> Optional[str]:
    """
    OpenAlex returns abstracts as abstract_inverted_index: {word: [pos1, pos2, ...]}.
    This reconstructs a best-effort plaintext abstract.
    """
    if not inv:
        return None

    pairs: List[tuple[int, str]] = []
    for word, positions in inv.items():
        if not positions:
            continue
        for p in positions:
            if isinstance(p, int):
                pairs.append((p, word))

    if not pairs:
        return None

    pairs.sort(key=lambda x: x[0])
    max_pos = pairs[-1][0]
    words = [""] * (max_pos + 1)
    for pos, w in pairs:
        if 0 <= pos <= max_pos:
            words[pos] = w

    text = " ".join(w for w in words if w).strip()
    return text or None


def venue_from_primary_location(work: Dict[str, Any]) -> Optional[str]:
    pl = work.get("primary_location") or {}
    src = pl.get("source") or {}
    return src.get("display_name")


def first_n_authors(work: Dict[str, Any], n: int = 6) -> str:
    authors = []
    for a in (work.get("authorships") or [])[:n]:
        name = ((a.get("author") or {}).get("display_name"))
        if name:
            authors.append(name)
    return "; ".join(authors)


def request_with_retries(
    url: str,
    params: Dict[str, Any],
    timeout: int = 30,
    max_retries: int = 5
) -> requests.Response:
    """
    Retries on transient errors (429, 5xx).
    """
    backoff = 1.0
    for attempt in range(1, max_retries + 1):
        r = requests.get(url, params=params, timeout=timeout)

        if r.status_code in (429, 500, 502, 503, 504):
            if attempt == max_retries:
                raise RuntimeError(f"OpenAlex error {r.status_code}\nURL: {r.url}\nBody: {r.text}")
            time.sleep(backoff)
            backoff *= 2
            continue

        if r.status_code >= 400:
            raise RuntimeError(f"OpenAlex error {r.status_code}\nURL: {r.url}\nBody: {r.text}")

        return r

    raise RuntimeError("Unexpected retry loop exit")


def fetch_works_for_query(q: str, max_works: Optional[int]) -> List[Dict[str, Any]]:
    """
    Fetches works for a query using cursor-based pagination.
    """
    rows: List[Dict[str, Any]] = []
    cursor = "*"

    select = (
        "id,display_name,publication_year,type,doi,cited_by_count,"
        "authorships,primary_location,abstract_inverted_index"
    )

    while cursor:
        params = {
            "search": q,
            "per-page": PER_PAGE,
            "cursor": cursor,
            "select": select,
            "api_key": OPENALEX_API_KEY,
        }

        r = request_with_retries(BASE_URL, params=params, timeout=TIMEOUT_SEC)
        data = r.json()

        for w in data.get("results", []) or []:
            rows.append({
                "query": q,
                "title": w.get("display_name"),
                "year": w.get("publication_year"),
                "type": w.get("type"),
                "venue": venue_from_primary_location(w),
                "cited_by": w.get("cited_by_count"),
                "authors(first6)": first_n_authors(w, n=6),
                "doi": w.get("doi"),
                "openalex_id": w.get("id"),
                "abstract": abstract_from_inverted_index(w.get("abstract_inverted_index")),
            })

            if max_works is not None and len(rows) >= max_works:
                return rows

        cursor = (data.get("meta") or {}).get("next_cursor")
        if not cursor:
            break

    return rows


# ----------------------------
# Run
# ----------------------------
all_rows: List[Dict[str, Any]] = []
for q in QUERY_LIST:
    all_rows.extend(fetch_works_for_query(q, max_works=MAX_WORKS_PER_QUERY))

df_openalex = pd.DataFrame(all_rows)

# De-dupe (OpenAlex ID is best unique key; DOI/title as backup)
if not df_openalex.empty:
    df_openalex = df_openalex.drop_duplicates(
        subset=["openalex_id", "doi", "title"],
        keep="first"
    ).reset_index(drop=True)

# Save to CSV
df_openalex.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8")
print(f"Saved {len(df_openalex):,} works to: {OUTPUT_CSV_PATH}")

df_openalex.head(50)

Saved 1,625 works to: openalex_works.csv


,query,title,year,type,venue,cited_by,authors(first6),doi,openalex_id,abstract
0,dynamic pricing definition concepts brick-and-...,Digital Transformation: An Overview of the Cur...,2021,article,SAGE Open,1044,Sascha Kraus; Paul Jones; Norbert Kailer; Alex...,https://doi.org/10.1177/21582440211047576,https://openalex.org/W3202582446,The increasing digitalization of economies has...
1,dynamic pricing definition concepts brick-and-...,Metaverse marketing: How the metaverse will sh...,2022,article,Psychology and Marketing,755,Yogesh K. Dwivedi; Laurie Hughes; Yichuan Wang...,https://doi.org/10.1002/mar.21767,https://openalex.org/W4313310852,Abstract The initial hype and fanfare from the...
2,dynamic pricing definition concepts brick-and-...,Consumer-driven e-commerce,2018,article,International Journal of Physical Distribution...,332,Stanley Frederick W.T. Lim; Xin Jin; Jagjit Si...,https://doi.org/10.1108/ijpdlm-02-2017-0081,https://openalex.org/W2579462724,Purpose The purpose of this paper is to re-exa...
3,dynamic pricing definition concepts brick-and-...,"The Evolution of Social Commerce: The People, ...",2012,article,Communications of the Association for Informat...,533,Chingning Wang; Ping Zhang,https://doi.org/10.17705/1cais.03105,https://openalex.org/W191206311,Social commerce is a form of commerce mediated...
4,dynamic pricing definition concepts brick-and-...,"Exploring Factors That Affect Usefulness, Ease...",2015,article,International Journal of Management & Informat...,299,Yoon C. Cho; Esen Sagynov,https://doi.org/10.19030/ijmis.v19i1.9086,https://openalex.org/W1548690256,Various studies have examined the effects of f...
5,dynamic pricing definition concepts brick-and-...,A Review of Impulse Buying Behavior,2013,review,International Journal of Marketing Studies,320,Ravi Shankar Bhakat; G. Muruganantham,https://doi.org/10.5539/ijms.v5n3p149,https://openalex.org/W2101151549,Researchers and Practitioners have been intere...
6,dynamic pricing definition concepts brick-and-...,A review of the environmental implications of ...,2015,review,International Journal of Physical Distribution...,236,Riccardo Mangiaracina; Gino Marchet; Sara Pero...,https://doi.org/10.1108/ijpdlm-06-2014-0133,https://openalex.org/W2165069623,Purpose – Given the importance of logistics op...
7,dynamic pricing definition concepts brick-and-...,Amazon's Antitrust Paradox,2017,article,Yale Law School Legal Scholarship Repository,205,Lina Khan,None,https://openalex.org/W3122148702,Amazon is the titan of twenty-first century co...
8,dynamic pricing definition concepts brick-and-...,Sustainable Online Shopping Logistics for Cust...,2019,article,Sustainability,113,Daeheon Choi; Chune Young Chung; Jason Young,https://doi.org/10.3390/su11205626,https://openalex.org/W2979639440,This study examines the impact of the quality ...
9,dynamic pricing definition concepts brick-and-...,To immerse or not? Experimenting with two virt...,2017,article,Information Technology and People,163,Savvas Papagiannidis; Eleonora Pantano; Eric W...,https://doi.org/10.1108/itp-03-2015-0069,https://openalex.org/W2311676256,Purpose The purpose of this paper is to examin...


# Semantic Scholar (Graph API) — relevance-ranked paper search

In [3]:
import os
import re
import json
import time
import random
import hashlib
from pathlib import Path
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, List, Optional, Iterable

import requests
import pandas as pd


# =============================
# USER CONFIG
# =============================
# Your existing list
QUERY_LIST = QUERY_LIST

API_KEY = os.getenv("S2_API_KEY", "").strip()  # recommended but optional

BASE = "https://api.semanticscholar.org/graph/v1"
SEARCH_URL = f"{BASE}/paper/search"          # relevance search (what you used originally)
BATCH_URL  = f"{BASE}/paper/batch"

# Performance toggles
FETCH_ABSTRACTS_VIA_BATCH = True   # search = light fields, then batch fetch abstracts
CACHE_ENABLED = True
CACHE_TTL_DAYS = 30

# How many results per query (relevance search supports offset+limit; limit typically <=100)
LIMIT = 100
MAX_PAGES_PER_QUERY = 1            # set to 2..10 if you want more than 100/query (up to 1000 total results)

# Batch details (paper/batch supports many IDs per call; 200 is safe)
BATCH_SIZE = 200

SAVE_CSV_PATH = "semantic_scholar_results.csv"


# =============================
# Fields (big speed win)
# =============================
# Keep search light (no abstract)
SEARCH_FIELDS = "paperId,title,year,authors,venue,citationCount,externalIds,url"
# Fetch abstract later
DETAIL_FIELDS = "paperId,abstract"


# =============================
# Session
# =============================
session = requests.Session()
session.headers.update({"User-Agent": "s2-relevance-fast/0.6"})
if API_KEY:
    session.headers.update({"x-api-key": API_KEY})

print(
    "Using API key:", bool(API_KEY),
    "| x-api-key header present:", "x-api-key" in session.headers,
    "| key length:", len(session.headers.get("x-api-key", "")),
)


# =============================
# Cache helpers
# =============================
CACHE_DIR = Path(".s2_cache")
CACHE_DIR.mkdir(exist_ok=True)

def _now_utc() -> datetime:
    return datetime.now(timezone.utc)

def _cache_key(method: str, url: str, params: Optional[Dict[str, Any]], body: Optional[Dict[str, Any]]) -> str:
    blob = {"m": method.upper(), "u": url, "p": params or {}, "b": body or {}}
    s = json.dumps(blob, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha1(s).hexdigest()

def cache_get(method: str, url: str, params: Optional[Dict[str, Any]], body: Optional[Dict[str, Any]]) -> Optional[Any]:
    if not CACHE_ENABLED:
        return None
    key = _cache_key(method, url, params, body)
    path = CACHE_DIR / f"{key}.json"
    if not path.exists():
        return None
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
        created = datetime.fromisoformat(payload["created"])
        if _now_utc() - created > timedelta(days=CACHE_TTL_DAYS):
            return None
        return payload["data"]
    except Exception:
        return None

def cache_set(method: str, url: str, params: Optional[Dict[str, Any]], body: Optional[Dict[str, Any]], data: Any) -> None:
    if not CACHE_ENABLED:
        return
    key = _cache_key(method, url, params, body)
    path = CACHE_DIR / f"{key}.json"
    payload = {"created": _now_utc().isoformat(), "data": data}
    path.write_text(json.dumps(payload), encoding="utf-8")


# =============================
# Adaptive pacer (reliable without wasting time)
# =============================
class Pacer:
    def __init__(self):
        # unauth traffic tends to get throttled more; start slower if no key
        self.min_interval = 1.05 if API_KEY else 2.5
        self.max_interval = 30.0
        self.interval = self.min_interval
        self.next_time = 0.0
        self.ok_streak = 0

    def wait(self):
        now = time.monotonic()
        if now < self.next_time:
            time.sleep(self.next_time - now)

    def _schedule(self, extra_wait: float = 0.0):
        self.next_time = time.monotonic() + self.interval + extra_wait + random.random() * 0.15

    def on_200(self):
        self.ok_streak += 1
        factor = 0.90 if self.ok_streak < 3 else 0.80
        self.interval = max(self.min_interval, self.interval * factor)
        self._schedule()

    def on_429(self, retry_after: Optional[float]):
        self.ok_streak = 0
        self.interval = min(self.max_interval, max(self.min_interval, self.interval * 1.5))
        self._schedule(extra_wait=retry_after or 0.0)

    def on_5xx(self, retry_after: Optional[float]):
        self.ok_streak = 0
        self.interval = min(self.max_interval, max(self.min_interval, self.interval * 1.2))
        self._schedule(extra_wait=retry_after or 0.0)

pacer = Pacer()

def _short_body(resp: requests.Response, n: int = 220) -> str:
    txt = (resp.text or "").strip().replace("\n", " ")
    return (txt[:n] + "…") if len(txt) > n else txt

def _parse_retry_after(resp: requests.Response) -> Optional[float]:
    ra = resp.headers.get("Retry-After")
    if not ra:
        return None
    try:
        return float(ra)
    except ValueError:
        return None


# =============================
# Request wrapper
# =============================
def request_json(method: str, url: str, params: Optional[Dict[str, Any]] = None,
                 body: Optional[Dict[str, Any]] = None, max_retries: int = 50, verbose: bool = True) -> Any:
    cached = cache_get(method, url, params, body)
    if cached is not None:
        if verbose:
            print(f"  🧠 cache hit: {method} {url}")
        return cached

    for attempt in range(1, max_retries + 1):
        pacer.wait()
        if verbose:
            print(f"  [S2] {method} attempt {attempt}/{max_retries} | pacer_interval={pacer.interval:.2f}s")

        try:
            resp = session.request(method, url, params=params, json=body, timeout=30)
        except requests.RequestException as e:
            extra = min(10.0, 0.5 * (2 ** (attempt - 1))) + random.random() * 0.25
            if verbose:
                print(f"    ↪ network error: {e} | extra wait {extra:.2f}s")
            pacer._schedule(extra_wait=extra)
            continue

        if verbose:
            print(f"    ↩ status={resp.status_code} | Retry-After={resp.headers.get('Retry-After')}")

        if resp.status_code == 200:
            pacer.on_200()
            data = resp.json()
            cache_set(method, url, params, body, data)
            return data

        retry_after = _parse_retry_after(resp)

        if resp.status_code == 429:
            if verbose:
                print(f"    ⚠️ 429 throttled | body: {_short_body(resp)}")
            pacer.on_429(retry_after)
            continue

        if resp.status_code in (500, 502, 503, 504):
            if verbose:
                print(f"    ⚠️ {resp.status_code} transient | body: {_short_body(resp)}")
            pacer.on_5xx(retry_after)
            continue

        if verbose:
            print(f"    ❌ non-retryable {resp.status_code} | body: {_short_body(resp, 500)}")
        resp.raise_for_status()

    raise RuntimeError(f"Giving up after {max_retries} retries: {method} {url}")


# =============================
# Utilities
# =============================
def clean_query(q: str) -> str:
    q = q.replace("-", " ")
    q = re.sub(r"\s+", " ", q).strip()
    return q

def chunked(xs: List[str], n: int) -> Iterable[List[str]]:
    for i in range(0, len(xs), n):
        yield xs[i:i + n]


# =============================
# Pipeline
# =============================
rows: List[Dict[str, Any]] = []

for qi, q in enumerate(QUERY_LIST, start=1):
    q2 = clean_query(q)
    print(f"\n=== ({qi}/{len(QUERY_LIST)}) RELEVANCE SEARCH: {q2!r} ===")

    offset = 0
    for page in range(1, MAX_PAGES_PER_QUERY + 1):
        params = {
            "query": q2,
            "fields": SEARCH_FIELDS,
            "limit": LIMIT,
            "offset": offset,
        }

        data = request_json("GET", SEARCH_URL, params=params, body=None, max_retries=50, verbose=True)
        papers = data.get("data", []) or []
        print(f"→ page {page}: got {len(papers)} papers")

        for p in papers:
            authors = [a.get("name") for a in (p.get("authors") or [])[:6] if a.get("name")]
            ext = p.get("externalIds") or {}
            rows.append({
                "query": q,
                "paperId": p.get("paperId"),
                "title": p.get("title"),
                "year": p.get("year"),
                "venue": p.get("venue"),
                "citationCount": p.get("citationCount"),
                "authors(first6)": "; ".join(authors),
                "doi": ext.get("DOI"),
                "s2_url": p.get("url"),
                # abstract later
            })

        if len(papers) < LIMIT:
            break
        offset += LIMIT


df = pd.DataFrame(rows)

# Dedup within query+paperId (keeps overlaps across different queries)
df = df.drop_duplicates(subset=["query", "paperId"]).reset_index(drop=True)

# Batch fetch abstracts after dedupe
if FETCH_ABSTRACTS_VIA_BATCH :
    unique_ids = [pid for pid in df["paperId"].dropna().unique().tolist() if pid]
    print(f"\n=== BATCH ABSTRACTS: {len(unique_ids)} unique ids ===")

    abstracts: Dict[str, Optional[str]] = {}

    for bi, ids in enumerate(chunked(unique_ids, BATCH_SIZE), start=1):
        print(f"--- batch {bi} | size={len(ids)} ---")
        params = {"fields": DETAIL_FIELDS}
        body = {"ids": ids}
        batch = request_json("POST", BATCH_URL, params=params, body=body, max_retries=50, verbose=True)

        if isinstance(batch, list):
            for p in batch:
                pid = p.get("paperId")
                if pid:
                    abstracts[pid] = p.get("abstract")
        else:
            for p in (batch.get("data", []) or []):
                pid = p.get("paperId")
                if pid:
                    abstracts[pid] = p.get("abstract")

    df["abstract"] = df["paperId"].map(abstracts)

# Final dedupe (don’t accidentally collapse across queries)
df = df.drop_duplicates(subset=["query", "paperId"]).reset_index(drop=True)

df.to_csv(SAVE_CSV_PATH, index=False)
print(f"\nDone. Rows: {len(df)} | Unique papers: {df['paperId'].nunique()}")
print(f"Saved: {SAVE_CSV_PATH}")

df.head(50)


Using API key: False | x-api-key header present: False | key length: 0

=== (1/6) RELEVANCE SEARCH: 'dynamic pricing definition concepts brick and mortar retail in store offline retail' ===
  🧠 cache hit: GET https://api.semanticscholar.org/graph/v1/paper/search
→ page 1: got 0 papers

=== (2/6) RELEVANCE SEARCH: 'dynamic pricing theory conceptual framework retail' ===
  🧠 cache hit: GET https://api.semanticscholar.org/graph/v1/paper/search
→ page 1: got 100 papers

=== (3/6) RELEVANCE SEARCH: 'time based pricing time dependent pricing dynamic pricing retail store' ===
  🧠 cache hit: GET https://api.semanticscholar.org/graph/v1/paper/search
→ page 1: got 100 papers

=== (4/6) RELEVANCE SEARCH: 'demand based pricing demand driven pricing dynamic pricing retail store' ===
  🧠 cache hit: GET https://api.semanticscholar.org/graph/v1/paper/search
→ page 1: got 100 papers

=== (5/6) RELEVANCE SEARCH: 'revenue management pricing retail dynamic pricing in store' ===
  🧠 cache hit: GET https://

,query,paperId,title,year,venue,citationCount,authors(first6),doi,s2_url,abstract
0,dynamic pricing theory conceptual framework re...,10b032f405418862ee6cbabac505542398709ffa,Dynamic Retail Pricing via Q-Learning - A Rein...,2025.0,2025 1st International Conference on AIML-Appl...,12,Mohit Apte; rd Pranav Datar; Ketan Kale; Dr P....,10.1109/ICAET63349.2025.10932302,https://www.semanticscholar.org/paper/10b032f4...,This study examines how a reinforcement learni...
1,dynamic pricing theory conceptual framework re...,9301266dbebcb2037adf55f09afbd3ad2106d742,Advanced Deep Reinforcement Learning Framework...,2024.0,International Conference on Computing Communic...,2,Anurag Agnihotri; I. Raj,10.1109/ICCCNT61001.2024.10725071,https://www.semanticscholar.org/paper/9301266d...,Pricing transparency is essential in online co...
2,dynamic pricing theory conceptual framework re...,82e42f51ff83bc1fa1a53025c4834dc68457470c,Reforming the innovation system to deliver aff...,2025.0,Journal of Pharmaceutical Policy and Practice,3,Suerie Moon; Adrian Alonso Ruiz; Marcela Vieir...,10.1080/20523211.2024.2436899,https://www.semanticscholar.org/paper/82e42f51...,ABSTRACT Background The current mainstream pha...
3,dynamic pricing theory conceptual framework re...,7a145bbcab2586e0a0b9681524056cf3e96c7669,Joint Optimization of Dynamic Pricing and Peri...,2025.0,2025 International Conference on Artificial In...,2,Anmol Aggarwal; Uttam Kumar; Sai Rakesh Reddy ...,10.1109/AIMV66517.2025.11203504,https://www.semanticscholar.org/paper/7a145bbc...,"Perishable goods, such as fresh produce or dai..."
4,dynamic pricing theory conceptual framework re...,4436d58d863c78ae2d04956b812fd2eb996e4fa9,A conceptual model for automation of product d...,2016.0,Kybernetes,4,Anup Kumar,10.1108/K-03-2015-0075,https://www.semanticscholar.org/paper/4436d58d...,None
5,dynamic pricing theory conceptual framework re...,0d2c7ed8ce1191e2507f3b10615b6174cffc5a38,Continuous dynamic pricing strategy under comp...,2024.0,Sādhanā,2,Anand Ranjan; J. K. Jha,10.1007/s12046-024-02468-1,https://www.semanticscholar.org/paper/0d2c7ed8...,None
6,dynamic pricing theory conceptual framework re...,d7953adedf4866a2a1142201aefce2f9557f6cd4,Dynamic Capabilities and MNE Global Strategy: ...,2023.0,Journal of Management Studies,74,Christos N. Pitelis; D. Teece; Hongyi Yang,10.1111/joms.13021,https://www.semanticscholar.org/paper/d7953ade...,Global strategy cannot be fully understood wit...
7,dynamic pricing theory conceptual framework re...,25617f1922f7f1f91ef235afa7b9f3c35b3a2f6a,Real Time Sales Forecasting in Omnichannel Ret...,2025.0,Academic Journal of Sociology and Management,3,Huanyu Liu; Tian Qi,10.70393/616a736d.323932,https://www.semanticscholar.org/paper/25617f19...,"In an omnichannel retail environment, accurate..."
8,dynamic pricing theory conceptual framework re...,bcdc65d4940486c2ec5b222d32b474f9de9b542c,Toward Dynamic Pricing for City-Wide Crowdsour...,2024.0,IEEE Transactions on Mobile Computing,5,Sen Bai; S. Tong; Xin Feng; Zhengang Jiang; Xi...,10.1109/TMC.2022.3228259,https://www.semanticscholar.org/paper/bcdc65d4...,The booming crowdsourced delivery leverages ne...
9,dynamic pricing theory conceptual framework re...,ccb8b9c457640cc3c5a51cb5c88e1ceb798488c2,Online Product Reviews-Triggered Dynamic Prici...,2019.0,Information systems research,89,Juan Feng; Xin Li; X. Zhang,10.1287/ISRE.2019.0852,https://www.semanticscholar.org/paper/ccb8b9c4...,Prior works offer compelling evidence that on ...


# Crossref (Works) — DOI-centric metadata across publishers

In [5]:
import requests
import pandas as pd

BASE_URL = "https://api.crossref.org/works"

# Crossref strongly prefers a descriptive User-Agent; include your email.
HEADERS = {
    "User-Agent": "chapter-sources-prototype/0.1 (mailto:your_email@example.com)"
}

rows = []
for q in QUERY_LIST:
    params = {
        "query.bibliographic": q,
        "rows": 100,
        "sort": "relevance",
        "order": "desc",
    }
    r = requests.get(BASE_URL, headers=HEADERS, params=params, timeout=30)
    r.raise_for_status()
    items = (r.json().get("message") or {}).get("items") or []

    for it in items:
        authors = []
        for a in (it.get("author") or [])[:6]:
            name = " ".join([x for x in [a.get("given"), a.get("family")] if x])
            if name:
                authors.append(name)

        title = (it.get("title") or [None])[0]
        year = None
        if it.get("issued", {}).get("date-parts"):
            year = it["issued"]["date-parts"][0][0]

        rows.append({
            "query": q,
            "title": title,
            "year": year,
            "type": it.get("type"),
            "container_title": (it.get("container-title") or [None])[0],
            "authors(first6)": "; ".join(authors),
            "doi": it.get("DOI"),
            "publisher": it.get("publisher"),
        })

df_crossref = pd.DataFrame(rows)
df_crossref = df_crossref.drop_duplicates(subset=["title", "doi"]).reset_index(drop=True)


# List the length of the dataframe
print(f"Total unique works retrieved: {len(df_crossref)}")

df_crossref.head(50)

Total unique works retrieved: 516


,query,title,year,type,container_title,authors(first6),doi,publisher
0,dynamic pricing definition concepts brick-and-...,In-Store Apps,2020.0,book-chapter,Retail Isn't Dead,Matthias Spanke,10.1007/978-3-030-36650-6_10,Springer International Publishing
1,dynamic pricing definition concepts brick-and-...,Saving Brick-and-Mortar Retail: Effects of Con...,2020.0,proceedings-article,Pivoting for the Pandemic,Ji Young Lee; Ki Ho Park,10.31274/itaa.12189,Iowa State University Digital Press
2,dynamic pricing definition concepts brick-and-...,Innovative Retail Retail User Behaviour of Pri...,2024.0,posted-content,None,Lingyao Jin,10.2139/ssrn.4849512,Elsevier BV
3,dynamic pricing definition concepts brick-and-...,On the Welfare Implications of Dynamic Pricing...,2017.0,journal-article,SSRN Electronic Journal,Ioannis Stamatopoulos; Achal Bassamboo,10.2139/ssrn.2927521,Elsevier BV
4,dynamic pricing definition concepts brick-and-...,Retail Isn't Dead,2020.0,book,None,Matthias Spanke,10.1007/978-3-030-36650-6,Springer International Publishing
5,dynamic pricing definition concepts brick-and-...,Saving Brick-and-Mortar Fashion Retail : The I...,2024.0,journal-article,International Journal of Costume and Fashion,Ji Young Lee; Ki Ho Park,10.7233/ijcf.2024.24.1.019,The Korea Society of Costume
6,dynamic pricing definition concepts brick-and-...,Delivery,2020.0,book-chapter,Retail Isn't Dead,Matthias Spanke,10.1007/978-3-030-36650-6_13,Springer International Publishing
7,dynamic pricing definition concepts brick-and-...,Community Hub,2020.0,book-chapter,Retail Isn't Dead,Matthias Spanke,10.1007/978-3-030-36650-6_4,Springer International Publishing
8,dynamic pricing definition concepts brick-and-...,Social Networks,2020.0,book-chapter,Retail Isn't Dead,Matthias Spanke,10.1007/978-3-030-36650-6_9,Springer International Publishing
9,dynamic pricing definition concepts brick-and-...,Virtual Reality,2020.0,book-chapter,Retail Isn't Dead,Matthias Spanke,10.1007/978-3-030-36650-6_6,Springer International Publishing


# 4) NCBI E-utilities (PubMed) — likely empty here, but good for biomedical chapters

In [ ]:
import os
import requests
import pandas as pd

NCBI_API_KEY = os.getenv("NCBI_API_KEY", "")  # optional

ESEARCH = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
ESUMMARY = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"

rows = []
for q in QUERY_LIST:
    params = {
        "db": "pubmed",
        "term": q,
        "retmax": 100,
        "retmode": "json",
    }
    if NCBI_API_KEY:
        params["api_key"] = NCBI_API_KEY

    r = requests.get(ESEARCH, params=params, timeout=30)
    r.raise_for_status()
    idlist = (r.json().get("esearchresult") or {}).get("idlist") or []
    if not idlist:
        continue

    params2 = {"db": "pubmed", "id": ",".join(idlist), "retmode": "json"}
    if NCBI_API_KEY:
        params2["api_key"] = NCBI_API_KEY

    r2 = requests.get(ESUMMARY, params=params2, timeout=30)
    r2.raise_for_status()
    result = r2.json().get("result") or {}
    uids = result.get("uids") or []

    for uid in uids:
        rec = result.get(uid) or {}
        authors = [a.get("name") for a in (rec.get("authors") or [])[:6] if a.get("name")]
        rows.append({
            "query": q,
            "pmid": uid,
            "title": rec.get("title"),
            "pubdate": rec.get("pubdate"),
            "source": rec.get("source"),
            "authors(first6)": "; ".join(authors),
        })

df_pubmed = pd.DataFrame(rows)
df_pubmed.head(50)


# arXiv API — preprints

In [ ]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET

BASE_URL = "http://export.arxiv.org/api/query"

def arxiv_search_query(q: str) -> str:
    # arXiv has its own query syntax; keep it simple for now:
    # search all fields for quoted phrase chunks
    # Example: all:"dynamic pricing" AND all:retail
    return f'all:"{q}"'

rows = []
for q in QUERY_LIST:
    params = {
        "search_query": arxiv_search_query(q),
        "start": 0,
        "max_results": 100,
    }
    r = requests.get(BASE_URL, params=params, timeout=30)
    r.raise_for_status()

    root = ET.fromstring(r.text)
    ns = {
        "atom": "http://www.w3.org/2005/Atom",
        "arxiv": "http://arxiv.org/schemas/atom",
    }

    for entry in root.findall("atom:entry", ns):
        title = (entry.findtext("atom:title", default="", namespaces=ns) or "").strip().replace("\n", " ")
        published = entry.findtext("atom:published", default=None, namespaces=ns)
        link = None
        for l in entry.findall("atom:link", ns):
            if l.attrib.get("rel") == "alternate":
                link = l.attrib.get("href")
        authors = [a.findtext("atom:name", default="", namespaces=ns) for a in entry.findall("atom:author", ns)]
        authors = [a for a in authors if a][:6]

        rows.append({
            "query": q,
            "title": title,
            "published": published,
            "authors(first6)": "; ".join(authors),
            "url": link,
        })

df_arxiv = pd.DataFrame(rows).drop_duplicates(subset=["title", "url"]).reset_index(drop=True)
df_arxiv.head(50)


# Open Library (Search API) — books

In [ ]:
import requests
import pandas as pd

BASE_URL = "https://openlibrary.org/search.json"

rows = []
for q in QUERY_LIST:
    params = {"q": q, "limit": 100, "offset": 0}
    r = requests.get(BASE_URL, params=params, timeout=30)
    r.raise_for_status()
    docs = r.json().get("docs") or []

    for d in docs:
        authors = (d.get("author_name") or [])[:6]
        rows.append({
            "query": q,
            "title": d.get("title"),
            "first_publish_year": d.get("first_publish_year"),
            "edition_count": d.get("edition_count"),
            "authors(first6)": "; ".join(authors),
            "openlibrary_key": d.get("key"),
        })

df_openlibrary = pd.DataFrame(rows).drop_duplicates(subset=["title", "openlibrary_key"]).reset_index(drop=True)
df_openlibrary.head(50)
